# Multi-Cancer Dataset Expansion
**Genomic-RawSeq-Analyzer — Semester 2**

Downloads and preprocesses two new WXS cohorts from NCBI SRA to test
the pipeline's generalizability across cancer types:

| Cohort | Type | GEO | Samples |
|--------|------|-----|---------|
| **BRCA** | Breast Invasive Carcinoma | GSE48215 | 25 Tumor + 25 Normal |
| **LUAD** | Lung Adenocarcinoma | GSE40419 | 17 Tumor + 13 Normal |

After download, evaluates the **Semester 1 CNN** on each new cohort **zero-shot**
(no retraining) to measure cross-cancer transfer performance.

**Steps:**
1. Install `sra-tools` and download FASTQ files via `fasterq-dump`
2. Preprocess using existing `data_loader.py`
3. Check class balance
4. Run zero-shot CNN evaluation (read-level + patient-level AUC)

**Outputs saved to Google Drive:**
- `results/multi_cancer/brca/` — batch .npz files
- `results/multi_cancer/luad/` — batch .npz files
- `results/multi_cancer/brca/zero_shot_eval/`
- `results/multi_cancer/luad/zero_shot_eval/`

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support

!pip install -q tensorflow biopython scikit-learn matplotlib

# The apt 'sra-toolkit' (2.11.3) ships an old fasterq-dump that lacks `-X`
# (read-count limiting), forcing full-run downloads (multi-GB, very slow).
# Install a recent static build from NCBI's GitHub releases instead — it
# supports `-X` so each download stays small (~tens of MB for 50k reads).
SRA_TOOLS_DIR = '/content/sratoolkit.3.1.1-ubuntu64'
if not os.path.exists(SRA_TOOLS_DIR):
    print('Installing modern sra-tools (3.1.1) with -X support...')
    !wget -q https://github.com/ncbi/sra-tools/releases/download/3.1.1/sratoolkit.3.1.1-ubuntu64.tar.gz
    !tar -xzf sratoolkit.3.1.1-ubuntu64.tar.gz
os.environ['PATH'] = f'{SRA_TOOLS_DIR}/bin:' + os.environ['PATH']

!which fasterq-dump && fasterq-dump --version || echo 'fasterq-dump not found'
from tensorflow.keras.models import load_model

BASE = '/content/drive/MyDrive/DNA_Anomaly_Detection'

# ── Inline: load_all_batches ──────────────────────────────────────────
def load_all_batches(batch_dir):
    files = sorted(glob.glob(os.path.join(batch_dir, 'batch_*.npz')))
    if not files:
        raise FileNotFoundError(f'No batch_*.npz files in {batch_dir}')
    X_parts, y_parts, id_parts = [], [], []
    for f in files:
        print(f'Loading {os.path.basename(f)}...')
        with np.load(f, allow_pickle=True) as d:
            X_parts.append(d['X'])
            y_parts.append(d['y'])
            if 'run_ids' in d:
                id_parts.append(d['run_ids'])
            else:
                n = len(d['X'])
                name = os.path.basename(f).replace('.npz', '')
                id_parts.append(np.array([f'{name}_read_{i}' for i in range(n)]))
    X = np.concatenate(X_parts)
    y = np.concatenate(y_parts)
    run_ids = np.concatenate(id_parts)
    print(f'Total: X={X.shape}  Tumor={int(y.sum()):,}  Normal={int((y==0).sum()):,}')
    return X, y, run_ids

# ── Inline: seq_to_int ────────────────────────────────────────────────
_ENC = {'A': 1, 'C': 2, 'G': 3, 'T': 4, 'N': 5}
def seq_to_int(seq, max_len=80):
    nums = [_ENC.get(b, 5) for b in seq[:max_len]]
    if len(nums) < max_len:
        nums += [0] * (max_len - len(nums))
    return nums

# ── Inline: encode_fastq_file ─────────────────────────────────────────
def encode_fastq_file(fastq_path, label, run_id, max_reads=50_000, max_len=80):
    from Bio import SeqIO
    X, y, ids = [], [], []
    with open(fastq_path) as f:
        for rec in SeqIO.parse(f, 'fastq'):
            X.append(seq_to_int(str(rec.seq), max_len))
            y.append(label)
            ids.append(run_id)
            if len(X) >= max_reads:
                break
    return X, y, ids

# ── Inline: class_balance_report ─────────────────────────────────────
def class_balance_report(batch_dir, cancer_label='Cancer'):
    X, y, _ = load_all_batches(batch_dir)
    n_tumor, n_normal, total = int(y.sum()), int((y==0).sum()), len(y)
    print(f'\nCLASS BALANCE — {cancer_label}')
    print(f'  Total: {total:,}  Tumor: {n_tumor:,}  Normal: {n_normal:,}')
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Normal', 'Tumor'], [n_normal, n_tumor], color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'Class Balance\n{cancer_label}', fontsize=12)
    ax.set_ylabel('Read Count')
    for i, v in enumerate([n_normal, n_tumor]):
        ax.text(i, v + total * 0.005, f'{v:,}', ha='center', fontsize=10)
    plt.tight_layout()
    save_path = os.path.join(batch_dir, 'class_balance.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

# ── Inline: zero_shot_eval ────────────────────────────────────────────
def zero_shot_eval(model_path, batch_dir, cancer_label='Cancer', save_dir=None):
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    print(f'Loading model from {model_path}...')
    model = load_model(model_path)
    X, y, run_ids = load_all_batches(batch_dir)
    print('Running inference...')
    probs = model.predict(X, batch_size=2048, verbose=1).flatten()
    fpr, tpr, _ = roc_curve(y, probs)
    read_auc = auc(fpr, tpr)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y, (probs >= 0.5).astype(int), average='binary', zero_division=0)
    df = pd.DataFrame({'run_id': run_ids, 'prob': probs, 'label': y})
    pat = df.groupby('run_id').agg(
        patient_prob=('prob', 'mean'),
        patient_label=('label', lambda x: int(x.mode()[0])),
    ).reset_index()
    pat_auc = None
    if len(pat['patient_label'].unique()) > 1:
        fpr_p, tpr_p, _ = roc_curve(pat['patient_label'], pat['patient_prob'])
        pat_auc = auc(fpr_p, tpr_p)
    print(f'\nZERO-SHOT — {cancer_label}')
    print(f'  Read AUC   : {read_auc:.4f}')
    if pat_auc:
        print(f'  Patient AUC: {pat_auc:.4f}')
    print(f'  Precision  : {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}')
    fig, axes = plt.subplots(1, 2 if pat_auc else 1,
                             figsize=(13 if pat_auc else 6, 5), dpi=130)
    if pat_auc is None:
        axes = [axes]
    axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC={read_auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',lw=1)
    axes[0].set_title(f'Read-Level ROC\n{cancer_label}', fontsize=12)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[0].legend(loc='lower right')
    if pat_auc:
        axes[1].plot(fpr_p, tpr_p, color='#2980b9', lw=2, label=f'AUC={pat_auc:.4f}')
        axes[1].plot([0,1],[0,1],'k--',lw=1)
        axes[1].set_title(f'Patient-Level ROC\n{cancer_label}', fontsize=12)
        axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
        axes[1].legend(loc='lower right')
    plt.tight_layout()
    if save_dir:
        path = os.path.join(save_dir, 'zero_shot_roc.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()
    return {'read_auc': read_auc, 'patient_auc': pat_auc, 'precision': prec, 'recall': rec, 'f1': f1}

# ── Inline: SRA table parser ──────────────────────────────────────────
def assign_labels_from_sra_table(table_path):
    df = pd.read_csv(table_path, sep=None, engine='python')
    df.columns = [c.strip() for c in df.columns]
    run_col = next((c for c in df.columns if c.lower() in ('run','run_id','sra_id')), None)
    if run_col is None:
        raise ValueError("SraRunTable has no 'Run' column.")
    text_cols = [c for c in df.columns if c.lower() in
                 ('sample_type','tissue_type','source_name','disease','tumor_normal','sample_description')]
    rows = []
    for _, row in df.iterrows():
        combined = ' '.join(str(row.get(c,'')) for c in text_cols).lower()
        if any(t in combined for t in ('tumor','cancer','malignant')):
            rows.append({'Run': row[run_col], 'Label': 1})
        elif any(t in combined for t in ('normal','healthy','adjacent')):
            rows.append({'Run': row[run_col], 'Label': 0})
    result = pd.DataFrame(rows)
    print(f'Parsed {len(result)} runs (tumor={result["Label"].sum()}, normal={(result["Label"]==0).sum()})')
    return result

# ── Cohort metadata ───────────────────────────────────────────────────
COHORT_METADATA = {
    'brca': {
        'cancer_label': 'Breast Invasive Carcinoma (BRCA)',
        'geo_accession': 'GSE48215', 'sra_project': 'SRP028580',
        'n_tumor': 25, 'n_normal': 25,
        'runs': {
            'SRR949537': 1, 'SRR949538': 1, 'SRR949539': 1, 'SRR949540': 1,
            'SRR949541': 0, 'SRR949542': 0, 'SRR949543': 0, 'SRR949544': 0,
        },
    },
    'luad': {
        'cancer_label': 'Lung Adenocarcinoma (LUAD)',
        'geo_accession': 'GSE40419', 'sra_project': 'SRP013469',
        'n_tumor': 17, 'n_normal': 13,
        'runs': {
            'SRR521456': 1, 'SRR521457': 1, 'SRR521458': 1, 'SRR521459': 1,
            'SRR521460': 0, 'SRR521461': 0, 'SRR521462': 0, 'SRR521463': 0,
        },
    },
}

# ── Configuration ─────────────────────────────────────────────────────
CANCER_TYPE    = 'brca'    # 'brca' or 'luad'
MAX_READS      = 50_000
CNN_MODEL_PATH = f'{BASE}/ML Models/BreastCancer_CNN_Model.keras'

meta       = COHORT_METADATA[CANCER_TYPE]
OUTPUT_DIR = f'{BASE}/results/multi_cancer/{CANCER_TYPE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup OK.')
print(f'Cohort  : {meta["cancer_label"]}')
print(f'GEO     : {meta["geo_accession"]}   SRA: {meta["sra_project"]}')
print(f'Expected: {meta["n_tumor"]} tumor + {meta["n_normal"]} normal')
print(f'Output  : {OUTPUT_DIR}')

## Configuration
Set `CANCER_TYPE` to `'brca'` or `'luad'`. Run this notebook twice to process both.

## Step 1 — Get SRA Run List
**Option A (recommended):** Download the SraRunTable from NCBI Run Selector and upload it.  
**Option B:** Use the small hardcoded subset in `multi_cancer_loader.py` (8 samples).

For Option A:
1. Go to: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP028580 (BRCA) or https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP013469 (LUAD)
2. Click **Metadata** → download `SraRunTable.txt`
3. Upload to this Colab session

In [2]:
SRA_TABLE_PATH = None   # set to '/content/SraRunTable.txt' if you uploaded it

if SRA_TABLE_PATH and os.path.exists(SRA_TABLE_PATH):
    run_df = assign_labels_from_sra_table(SRA_TABLE_PATH)
    # Filter to WXS only
    if 'LibraryStrategy' in run_df.columns:
        run_df = run_df[run_df['LibraryStrategy'] == 'WXS']
    print(f'Using SraRunTable: {len(run_df)} runs')
else:
    # Fallback: hardcoded subset
    run_df = pd.DataFrame([
        {'Run': acc, 'Label': lbl}
        for acc, lbl in meta['runs'].items()
    ])
    print(f'Using hardcoded subset: {len(run_df)} runs')
    print('TIP: Upload SraRunTable.txt for the full cohort.')

print(f'\nRuns to download: {len(run_df)}')
print(f'  Tumor  : {(run_df["Label"]==1).sum()}')
print(f'  Normal : {(run_df["Label"]==0).sum()}')
display(run_df.head(10))

Using hardcoded subset: 8 runs
TIP: Upload SraRunTable.txt for the full cohort.

Runs to download: 8
  Tumor  : 4
  Normal : 4


,Run,Label
0,SRR949537,1
1,SRR949538,1
2,SRR949539,1
3,SRR949540,1
4,SRR949541,0
5,SRR949542,0
6,SRR949543,0
7,SRR949544,0


## Step 2 — Download FASTQ Files via fasterq-dump
Each sample downloads ~500 MB. With 8 samples this takes ~10-15 min on Colab.

In [ ]:
FASTQ_DIR = f'/content/fastq_{CANCER_TYPE}'
os.makedirs(FASTQ_DIR, exist_ok=True)

# SRA-toolkit needs one-time non-interactive config, otherwise fasterq-dump
# fails immediately with exit code 16384 ("could not resolve accession").
import subprocess
subprocess.run(['vdb-config', '--set', '/repository/user/main/public/root=/content/sra_cache'],
               capture_output=True, text=True)
subprocess.run(['vdb-config', '--set', '/repository/user/cache-disabled=true'],
               capture_output=True, text=True)
os.makedirs('/content/sra_cache', exist_ok=True)

print(f'Downloading {len(run_df)} FASTQ files to {FASTQ_DIR}...')
for i, row in run_df.iterrows():
    acc = row['Run']
    outfile = os.path.join(FASTQ_DIR, f'{acc}_1.fastq')
    if os.path.exists(outfile):
        print(f'  {acc} already downloaded, skipping.')
        continue
    print(f'  [{i+1}/{len(run_df)}] Downloading {acc}...')
    proc = subprocess.run(
        ['fasterq-dump', acc, '--outdir', FASTQ_DIR,
         '--split-files', '--threads', '4', '--progress',
         '-X', str(MAX_READS)],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(f'    WARNING: fasterq-dump exited with code {proc.returncode} for {acc}')
        print(f'    --- stderr ---\n{proc.stderr.strip()[-1500:]}')
        print(f'    --- stdout ---\n{proc.stdout.strip()[-500:]}')

fastq_files = [f for f in os.listdir(FASTQ_DIR) if f.endswith('.fastq')]
print(f'\nDownloaded {len(fastq_files)} FASTQ files.')

## Step 3 — Preprocess: Integer-Encode & Save Batches

In [4]:
print(f'Processing {len(run_df)} samples into integer-encoded batches...')
FASTQ_DIR_LOCAL = f'/content/fastq_{CANCER_TYPE}'
batch_num = 1
X_buf, y_buf, id_buf = [], [], []

for i, row in run_df.iterrows():
    acc   = row['Run']
    label = int(row['Label'])
    fastq_path = os.path.join(FASTQ_DIR_LOCAL, f'{acc}_1.fastq')
    if not os.path.exists(fastq_path):
        print(f'  WARNING: {fastq_path} not found, skipping.')
        continue
    print(f'  Encoding {acc}  (label={label})...')
    X_r, y_r, id_r = encode_fastq_file(fastq_path, label, acc, max_reads=MAX_READS)
    X_buf.extend(X_r); y_buf.extend(y_r); id_buf.extend(id_r)

if X_buf:
    save_path = os.path.join(OUTPUT_DIR, f'batch_{batch_num:03d}.npz')
    np.savez_compressed(save_path,
                        X=np.array(X_buf, dtype=np.int8),
                        y=np.array(y_buf, dtype=np.int8),
                        run_ids=np.array(id_buf, dtype='U20'))
    print(f'Saved {len(X_buf):,} reads → {save_path}')
else:
    print('No reads encoded. Check that FASTQ files downloaded correctly.')

Processing 8 samples into integer-encoded batches...
No reads encoded. Check that FASTQ files downloaded correctly.


## Step 4 — Class Balance Report

In [ ]:
if glob.glob(os.path.join(OUTPUT_DIR, 'batch_*.npz')):
    class_balance_report(OUTPUT_DIR, cancer_label=meta['cancer_label'])
else:
    print(f'No batch_*.npz files in {OUTPUT_DIR} yet — '
          f'Step 2/3 (download/preprocess) must succeed first. Skipping.')

## Step 5 — Zero-Shot CNN Evaluation
Evaluates the Semester 1 CNN (trained on WXS breast cancer) on the new cohort
**without any retraining** to measure cross-cancer transfer performance.

In [ ]:
if glob.glob(os.path.join(OUTPUT_DIR, 'batch_*.npz')):
    eval_dir = f'{OUTPUT_DIR}/zero_shot_eval'
    zero_shot_eval(
        model_path=CNN_MODEL_PATH,
        batch_dir=OUTPUT_DIR,
        cancer_label=meta['cancer_label'],
        save_dir=eval_dir,
    )
    print(f'\nZero-shot evaluation plots saved to: {eval_dir}')
else:
    print(f'No batch_*.npz files in {OUTPUT_DIR} yet — '
          f'Step 2/3 (download/preprocess) must succeed first. Skipping.')

## Summary

| Metric | WXS Breast (Sem. 1) | New Cohort (zero-shot) |
|--------|--------------------|-----------------------|
| Read-level AUC | 0.6157 | *see output above* |
| Patient-level AUC | 0.9156 | *see output above* |

**Interpretation:**
- If zero-shot AUC > 0.55: the CNN learned transferable somatic mutation features
- If zero-shot AUC ≈ 0.50: the signal is cancer-type specific → fine-tuning needed
- Patient-level crowd-voting will still amplify any consistent signal